# Demo Generative AI LLM bot

There are many Generative AI models and systems. We could try to get a connection to one of the big AI company AIs, but they may require payment for tokens (e.g., Claude requires a minimum of $5 to [get an API key from here](https://platform.claude.com/dashboard)).

Instead we can download free models and run them ourselves. You can search for downloadable models from [Hugging Face](https://huggingface.co/).

We will use a small Large Language Model (LLM) called [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct).

## Download and install the LLM (SmolLM2)

First we need a library called `transformers` for working with LLMS

In [ ]:
!pip install transformers

Next we download one of the LLMs (we'll use SmolLM2)

Note: There are different options for size and quality of the LLM

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Choose one of these models to download by uncommenting out that line

#checkpoint = "HuggingFaceTB/SmolLM2-135M-Instruct"
checkpoint = "HuggingFaceTB/SmolLM2-360M-Instruct"
#checkpoint = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
#checkpoint = "HuggingFaceTB/SmolLM3-3B"

# see website for model: https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct

# For jupyterHub we can't use the faster gpu (device = "cuda")
# so we need to set it to use the slower cpu
device = "cpu"

## Helper function for doing a request to the LLM
This is a helper function that takes instructions for the LLM (system_instructions_prompt) and the user prompt (user_prompt) to reply to

In [ ]:
def runLlmPrompt(system_instructions_prompt, user_prompt, max_new_tokens=50, show_steps=False):
    # Start new LLM session (I think)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    # for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
    model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)
    


    messages = [{"role": "system", "content": system_instructions_prompt},
                {"role": "user", "content": user_prompt}]
    input_text=tokenizer.apply_chat_template(messages, tokenize=False)

    if show_steps:
        print("Applying user text:")
        print()
        print(input_text)
        print("----------")
    
    inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
    outputs = model.generate(inputs, max_new_tokens=max_new_tokens, temperature=0.2, top_p=0.9, do_sample=True)
    full_output_text = tokenizer.decode(outputs[0])

    if show_steps:
        print("full chat text:")
        print()
        print(full_output_text)
        print("----------")
    
    # get the output
    output_response = output_response = full_output_text.split("\n<|im_start|>assistant\n")[1].replace("<|im_end|>", "")
    if show_steps:
        print("model output:")
        print()
        print(output_response)

    return output_response

## Test LLM (talk like a pirate)

In [ ]:
pirate_instructions = "You are a pirate. Respond with the most piratey words and phrases possible."

In [ ]:
runLlmPrompt(
    pirate_instructions,
    "Hello, how are you?"
)

In [ ]:
runLlmPrompt(
    pirate_instructions,
    "The weather sure is nice today!"
)

## Demo Pirate Bot

### Log into BlueSky

In [ ]:
# Load some code called "Client" from the "atproto" library that will help us work with Bluesky
from atproto import Client

In [ ]:
# Login to Bluesky
%run bluesky_keys.py

client = Client(base_url="https://bsky.social")
client.login(handle, password)

### Get My Latest Mention

In [ ]:
# look up my Bluesky handle
my_handle = client.me.handle

print("my handle is:" + my_handle)

In [ ]:
# search posts that mention my handle
mentions = client.app.bsky.feed.search_posts({
    'q': my_handle, 
    'mentions': my_handle
}).posts

if(len(mentions) < 1):
    print("Error: Could not find matching mentions!")
    
latest_mention = mentions[0]

# display the text of the post, so we can see what we found
display("latest mention post text: " + str(latest_mention.record.text))

### Get a pirate response

In [ ]:
pirate_instructions = "You are a pirate. Respond with the most piratey words and phrases possible."

pirate_response = runLlmPrompt(
    pirate_instructions,
    latest_mention.record.text
)

print("pirate response text: " + pirate_response)

### Reply to mention

In [ ]:

# make sure response isn't too long
response = pirate_response[:299]

client.send_post(
    response, 
    reply_to={'root': {'uri': latest_mention.uri, 'cid': latest_mention.cid}, 
              'parent': {'uri': latest_mention.uri, 'cid': latest_mention.cid}}
)

## Demo Disagreement Bot
When a user tags our bot, this code takes whatever the user says and then uses an LLM to make a reply by agreeing with the user.

This is a malicious bot, though it only replies to people who message our bot with it's disagreement, real bots on Bluesky [have been found](https://pivot-to-ai.com/2024/12/07/a-polite-disagreement-bot-ring-is-flooding-bluesky-reply-guy-as-a-disservice/) that spam popular posts by politely disagreeing with any popular posts they could find. This is also similar to how Russia has tried to disrupt US politics and society by trying to get Americans upset at each other over politics or even pop culture (e.g., [Star Wars movies](https://www.wired.com/story/star-wars-russian-trolls-study/)).

Also related to this is the joke "[disc horse bot](https://bsky.app/profile/horsedisc.bsky.social)" (sounds like the word "discourse," meaning "what are people arguing about currently"). The "disk horse bot" automatically creates invented controversial statements like "Eating glue is the weaponization of democracy."

### Log into BlueSky

In [ ]:
# Load some code called "Client" from the "atproto" library that will help us work with Bluesky
from atproto import Client

In [ ]:
# Login to Bluesky
%run bluesky_keys.py

client = Client(base_url="https://bsky.social")
client.login(handle, password)

### Get My Latest Mention

In [ ]:
# look up my Bluesky handle
my_handle = client.me.handle

print("my handle is:" + my_handle)

In [ ]:
# search posts that mention my handle
mentions = client.app.bsky.feed.search_posts({
    'q': my_handle, 
    'mentions': my_handle
}).posts

if(len(mentions) < 1):
    print("Error: Could not find matching mentions!")
    
latest_mention = mentions[0]

# display the text of the post, so we can see what we found
display("latest mention post text: " + str(latest_mention.record.text))

In [ ]:
latest_mention.record.text = "I like broccoli"

### Get response that agrees

In [ ]:
agreement_instructions = """
    You are a disagreement bot whose and you have an opinion, which is the opposite as whatever the user said. 
    It is extremely important that you politely disagree with whatever the user says, and tell them why!
"""

agreement_response = runLlmPrompt(
    agreement_instructions,
    latest_mention.record.text
)

print("agreement response text: " + agreement_response)

### Respond to the post that mentioned me

In [ ]:

# make sure response isn't too long
response = agreement_response[:299]

client.send_post(
    response, 
    reply_to={'root': {'uri': latest_mention.uri, 'cid': latest_mention.cid}, 
              'parent': {'uri': latest_mention.uri, 'cid': latest_mention.cid}}
)